# Family Quant AI - Data Explorer

Interactive notebook for exploring our market data, earnings, macro indicators, and fundamentals.

**Run `python update_market_data.py` first to ensure the database is populated.**

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

conn = duckdb.connect('../data/market_data.duckdb', read_only=True)

tables = conn.execute('SHOW TABLES').fetchdf()
print('Tables in database:')
for t in tables['name']:
    count = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {count:,} rows')

## 1. Stock Price History

In [ ]:
tsla = conn.execute("""
    SELECT timestamp::DATE as date, close
    FROM daily_bars WHERE symbol = 'TSLA' ORDER BY timestamp
""").fetchdf()

plt.figure(figsize=(14, 5))
plt.plot(tsla['date'], tsla['close'])
plt.title('TSLA Adjusted Close Price (Daily)')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Date range: {tsla['date'].min()} to {tsla['date'].max()}")
print(f"Total return: {(tsla['close'].iloc[-1] / tsla['close'].iloc[0] - 1) * 100:.1f}%")

In [ ]:
# Compare all tickers normalized to 100
all_prices = conn.execute("""
    SELECT symbol, timestamp::DATE as date, close
    FROM daily_bars ORDER BY symbol, timestamp
""").fetchdf()

plt.figure(figsize=(14, 6))
for sym in ['TSLA', 'AAPL', 'NVDA', 'SPY']:
    data = all_prices[all_prices['symbol'] == sym].copy()
    data['normalized'] = data['close'] / data['close'].iloc[0] * 100
    plt.plot(data['date'], data['normalized'], label=sym)

plt.title('All Tickers: Normalized Performance (Start = 100)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Annualized Returns

Uses **365 calendar days** (not 252 trading days) for fair comparison to CDs/bonds.

In [ ]:
returns = conn.execute("""
    SELECT symbol,
        MIN(timestamp)::DATE as start_date, MAX(timestamp)::DATE as end_date,
        FIRST(close ORDER BY timestamp) as first_close,
        LAST(close ORDER BY timestamp) as last_close
    FROM daily_bars GROUP BY symbol
""").fetchdf()

returns['total_return'] = returns['last_close'] / returns['first_close'] - 1
returns['calendar_days'] = (pd.to_datetime(returns['end_date']) - pd.to_datetime(returns['start_date'])).dt.days
returns['annualized'] = (1 + returns['total_return']) ** (365 / returns['calendar_days']) - 1

print('Annualized Returns (comparable to CD rates):')
print('=' * 50)
for _, row in returns.iterrows():
    print(f"  {row['symbol']}: {row['annualized']*100:+.1f}% per year  (total: {row['total_return']*100:+.1f}%)")

## 3. Earnings Surprises and Price Moves

In [ ]:
tsla_earnings = conn.execute("""
    SELECT e.reported_date, e.surprise_pct,
        (SELECT close FROM daily_bars WHERE symbol='TSLA' AND timestamp::DATE <= e.reported_date ORDER BY timestamp DESC LIMIT 1) as close_before,
        (SELECT close FROM daily_bars WHERE symbol='TSLA' AND timestamp::DATE > e.reported_date ORDER BY timestamp LIMIT 1) as close_after
    FROM earnings e
    WHERE e.symbol = 'TSLA' AND e.reported_date >= '2020-01-01'
    ORDER BY e.reported_date
""").fetchdf()

tsla_earnings['price_move_pct'] = (tsla_earnings['close_after'] / tsla_earnings['close_before'] - 1) * 100

print('TSLA: Stock Move After Each Earnings Report')
print('=' * 60)
for _, row in tsla_earnings.iterrows():
    d = '+' if row['price_move_pct'] > 0 else ''
    print(f"  {row['reported_date']}  Surprise: {row['surprise_pct']:+.1f}%  Price: {d}{row['price_move_pct']:.1f}%")

## 4. Macro Economic Indicators

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (sid, title) in zip(axes.flat, [
    ('FEDFUNDS', 'Fed Funds Rate (%)'),
    ('CPIAUCSL', 'CPI'),
    ('UNRATE', 'Unemployment (%)'),
    ('DGS10', '10-Year Treasury (%)'),
]):
    data = conn.execute(f"""
        SELECT observation_date, value FROM macro_releases
        WHERE series_id = '{sid}' AND observation_date >= '2016-01-01'
        ORDER BY observation_date
    """).fetchdf()
    ax.plot(data['observation_date'], data['value'])
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. SEC Fundamentals

In [ ]:
tsla_fund = conn.execute("""
    SELECT fiscal_date_ending, revenue/1e9 as revenue_B, net_income/1e9 as net_income_B
    FROM fundamentals WHERE symbol='TSLA' AND revenue IS NOT NULL
    ORDER BY fiscal_date_ending
""").fetchdf()

if not tsla_fund.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(range(len(tsla_fund)), tsla_fund['revenue_B'], alpha=0.7, label='Revenue')
    ax.plot(range(len(tsla_fund)), tsla_fund['net_income_B'], 'r-o', label='Net Income')
    ax.set_title('TSLA Quarterly Revenue and Net Income ($B)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
conn.close()
print('Done.')